# Gold: hospital_daily_summary

Builds `gold.hospital_daily_summary` (one row per hospital per day) from silver `lab_results`, enriched with `hospitals` and the calendar dimension, with LAG-based day-over-day test volume trend, per the STTM `sttm_silver_to_gold.xlsx`. Load strategy: full rebuild (CREATE OR REPLACE TABLE ... AS SELECT).

Notes: silver sources are deduplicated to their latest record per key before aggregation; the calendar dimension lives in the gold schema (`calendar`), not silver.

In [0]:
import re

# <catalog>/silver/gold placeholders from the STTM, mapped to this workspace's Unity Catalog names
dbutils.widgets.text("catalog", "man_catlog")
dbutils.widgets.text("silver_schema", "caresync_schema_silver")
dbutils.widgets.text("gold_schema", "caresync_schema_gold")

def _ident(name):
    v = dbutils.widgets.get(name).strip()
    if not re.fullmatch(r"[A-Za-z0-9_]+", v):
        raise ValueError(f"Invalid identifier for {name}: {v!r}")
    return v

catalog = _ident("catalog")
silver = f"{catalog}.{_ident('silver_schema')}"
gold = f"{catalog}.{_ident('gold_schema')}"
calendar_table = f"{gold}.calendar"   # calendar dimension was built in the gold schema
target_table = f"{gold}.hospital_daily_summary"
print(f"Source: {silver} | Calendar: {calendar_table} | Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per hospital per day with lab-test volume, quality rates and day-over-day trend (STTM silver->gold v1.0)'
AS
WITH lab_results AS (
  -- Silver holds duplicate rows per key; keep the latest record per lab result
  SELECT * FROM {silver}.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id ORDER BY reported_at DESC, landing_ts DESC) = 1
),
hospitals AS (
  SELECT * FROM {silver}.hospitals
  QUALIFY ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY updated_at DESC, landing_ts DESC) = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM lab_results
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  CAST(t.hospital_id AS INT)                                                        AS hospital_id,
  h.hospital_name,
  CAST(t.summary_date AS DATE)                                                      AS summary_date,
  c.day_name,
  c.is_weekend,
  CAST(t.total_tests AS INT)                                                        AS total_tests,
  CAST(t.unique_patients_tested AS INT)                                             AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                                AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                                AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                                AS critical_test_count,
  CAST(ROUND(t.abnormal_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2))     AS abnormal_rate_pct,
  CAST(ROUND(t.critical_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2))     AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                                               AS prev_day_total_tests,
  CAST(ROUND((t.total_tests - t.prev_day_total_tests) * 100.0
             / NULLIF(t.prev_day_total_tests, 0), 2) AS DECIMAL(6,2))               AS test_volume_change_pct
FROM trend t
LEFT JOIN hospitals h      ON t.hospital_id = h.hospital_id
LEFT JOIN {calendar_table} c ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT)
""")
print(f"Created {target_table}")

In [0]:
chk = spark.sql(f"""
SELECT
  (SELECT COUNT(*) FROM {target_table})                                             AS gold_rows,
  (SELECT COUNT(*) FROM (SELECT DISTINCT hospital_id, summary_date FROM {target_table})) AS gold_distinct_keys,
  (SELECT SUM(total_tests) FROM {target_table})                                     AS gold_total_tests,
  (SELECT COUNT(DISTINCT lab_result_id) FROM {silver}.lab_results)                  AS silver_distinct_tests,
  (SELECT COUNT_IF(day_name IS NULL) FROM {target_table})                           AS rows_missing_calendar
""").first()
display(spark.createDataFrame([chk]))
assert chk.gold_rows == chk.gold_distinct_keys, "Grain violated: expected one row per hospital per day"
assert chk.gold_total_tests == chk.silver_distinct_tests, "Lab test totals do not reconcile with silver"
print("Validation passed")

In [0]:
display(spark.table(target_table).orderBy("hospital_id", "summary_date").limit(20))